# 第 29 章：星系形态分类案例

这个 notebook 用一个 Galaxy Zoo 风格的教学数据集串起一个更接近真实项目的最小 workflow：

- 先做标签可信度和图像质量筛选
- 用传统形态学特征建立一个 baseline 分类器
- 观察哪些对象容易落在 spiral / elliptical / edge-on 的边界上
- 回到错分样本，判断它更像标签噪声、投影效应还是低质量观测

教学说明：这里使用的是从教学图像中预先提取好的形态学特征，而不是原始像素 cutout。这样可以在不额外引入图像处理依赖的前提下，先把 baseline、质量控制和错分复核讲清楚。


In [ ]:
from __future__ import annotations

import csv
import math
import os
from pathlib import Path

DATA_PATH = Path("../../data/small/galaxy_morphology_case_demo.csv").resolve()

NUMERIC_FIELDS = {
    "seeing_arcsec",
    "snr_per_pixel",
    "axis_ratio",
    "concentration",
    "asymmetry",
    "smoothness",
    "gini",
    "m20",
    "spiral_arm_strength",
    "edge_on_score",
    "vote_fraction",
}

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        row = {}
        for key, value in raw.items():
            row[key] = float(value) if key in NUMERIC_FIELDS else value
        rows.append(row)

print(f"Loaded {len(rows)} morphology samples from {DATA_PATH.name}")

class_counts = {}
quality_counts = {}
for row in rows:
    class_counts[row["morphology_label"]] = class_counts.get(row["morphology_label"], 0) + 1
    quality_counts[row["quality_flag"]] = quality_counts.get(row["quality_flag"], 0) + 1

print("class counts before cuts:", class_counts)
print("quality counts:", quality_counts)
print("first row:", rows[0])


In [ ]:
def count_by_label(sample_rows):
    counts = {}
    for row in sample_rows:
        counts[row["morphology_label"]] = counts.get(row["morphology_label"], 0) + 1
    return counts

analysis_rows = [
    row for row in rows
    if row["quality_flag"] == "clean" and row["vote_fraction"] >= 0.70
]
dropped_rows = [
    (row["galaxy_id"], row["quality_flag"], row["vote_fraction"])
    for row in rows
    if row not in analysis_rows
]

print("analysis sample size:", len(analysis_rows))
print("class counts after cuts:", count_by_label(analysis_rows))
print("dropped rows:", dropped_rows)
print(
    "seeing range [arcsec]:",
    (round(min(row["seeing_arcsec"] for row in analysis_rows), 2), round(max(row["seeing_arcsec"] for row in analysis_rows), 2)),
)
print(
    "snr range per pixel:",
    (round(min(row["snr_per_pixel"] for row in analysis_rows), 1), round(max(row["snr_per_pixel"] for row in analysis_rows), 1)),
)


In [ ]:
feature_names = [
    "axis_ratio",
    "concentration",
    "asymmetry",
    "smoothness",
    "gini",
    "m20",
    "spiral_arm_strength",
    "edge_on_score",
]

summary_by_label = {}
for row in analysis_rows:
    summary_by_label.setdefault(row["morphology_label"], {feature_name: [] for feature_name in feature_names})
    for feature_name in feature_names:
        summary_by_label[row["morphology_label"]][feature_name].append(row[feature_name])

print("feature ranges by class:")
for label, values in summary_by_label.items():
    compact_summary = {
        feature_name: (round(min(series), 3), round(max(series), 3))
        for feature_name, series in values.items()
        if feature_name in {"axis_ratio", "concentration", "spiral_arm_strength", "edge_on_score"}
    }
    print(label, compact_summary)

print("Teaching note:")
print("  spiral objects keep stronger spiral-arm indicators and lower concentration.")
print("  elliptical objects are centrally concentrated and structurally smooth.")
print("  edge-on disks collapse to small axis ratios and large edge-on scores.")


In [ ]:
feature_means = {}
feature_stds = {}
for feature_name in feature_names:
    values = [row[feature_name] for row in analysis_rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    feature_means[feature_name] = mean_value
    feature_stds[feature_name] = math.sqrt(variance) or 1.0

for row in analysis_rows:
    row["morphology_vector"] = [
        (row[feature_name] - feature_means[feature_name]) / feature_stds[feature_name]
        for feature_name in feature_names
    ]

print("standardized features ready:", feature_names)
print("means:", {name: round(value, 3) for name, value in feature_means.items()})
print("stds:", {name: round(value, 3) for name, value in feature_stds.items()})


In [ ]:
test_ids = {"SP05", "SP06", "EL04", "EL05", "EG04", "EG05"}
train_rows = [row for row in analysis_rows if row["galaxy_id"] not in test_ids]
test_rows = [row for row in analysis_rows if row["galaxy_id"] in test_ids]


def squared_distance(left_vector, right_vector):
    return sum((left - right) ** 2 for left, right in zip(left_vector, right_vector))


class_centroids = {}
for label in sorted({row["morphology_label"] for row in train_rows}):
    group_vectors = [row["morphology_vector"] for row in train_rows if row["morphology_label"] == label]
    class_centroids[label] = [
        sum(values[index] for values in group_vectors) / len(group_vectors)
        for index in range(len(feature_names))
    ]

prediction_rows = []
for row in test_rows:
    ranked_distances = sorted(
        (squared_distance(row["morphology_vector"], centroid), label)
        for label, centroid in class_centroids.items()
    )
    best_distance, predicted_label = ranked_distances[0]
    second_distance, runner_up_label = ranked_distances[1]
    prediction_rows.append({
        "galaxy_id": row["galaxy_id"],
        "true_label": row["morphology_label"],
        "predicted_label": predicted_label,
        "runner_up_label": runner_up_label,
        "distance_margin": second_distance - best_distance,
        "vote_fraction": row["vote_fraction"],
        "axis_ratio": row["axis_ratio"],
        "spiral_arm_strength": row["spiral_arm_strength"],
        "edge_on_score": row["edge_on_score"],
    })

confusion = {}
for row in prediction_rows:
    confusion.setdefault(row["true_label"], {})
    confusion[row["true_label"]][row["predicted_label"]] = confusion[row["true_label"]].get(row["predicted_label"], 0) + 1

recall_by_class = {}
for label, counts in confusion.items():
    recall_by_class[label] = round(counts.get(label, 0) / sum(counts.values()), 3)

accuracy = sum(row["true_label"] == row["predicted_label"] for row in prediction_rows) / len(prediction_rows)

print("train class counts:", count_by_label(train_rows))
print("test predictions:")
for row in prediction_rows:
    print(
        row["galaxy_id"],
        row["true_label"],
        "->",
        row["predicted_label"],
        "runner_up=",
        row["runner_up_label"],
        "margin=",
        round(row["distance_margin"], 3),
    )
print("test accuracy:", round(accuracy, 3))
print("confusion matrix:", confusion)
print("recall by class:", recall_by_class)


In [ ]:
misclassified_rows = [row for row in prediction_rows if row["true_label"] != row["predicted_label"]]
uncertain_rows = sorted(prediction_rows, key=lambda row: row["distance_margin"])

print("misclassified rows:", misclassified_rows)
print("lowest-margin test rows:", [
    (row["galaxy_id"], row["true_label"], row["predicted_label"], round(row["distance_margin"], 3))
    for row in uncertain_rows[:3]
])


def nearest_training_neighbors(target_row, candidate_rows, neighbor_count=3):
    distances = []
    for candidate_row in candidate_rows:
        distances.append((
            math.sqrt(squared_distance(target_row["morphology_vector"], candidate_row["morphology_vector"])),
            candidate_row["galaxy_id"],
            candidate_row["morphology_label"],
            candidate_row["quality_flag"],
        ))
    distances.sort(key=lambda item: item[0])
    return distances[:neighbor_count]

for target_id in ["SP06", "EL05"]:
    target_row = next(row for row in analysis_rows if row["galaxy_id"] == target_id)
    neighbors = nearest_training_neighbors(target_row, train_rows)
    print(target_id, "nearest training neighbors:", [
        (round(distance, 3), galaxy_id, label, quality_flag)
        for distance, galaxy_id, label, quality_flag in neighbors
    ])
    print(target_id, "feature snapshot:", {
        "axis_ratio": round(target_row["axis_ratio"], 3),
        "concentration": round(target_row["concentration"], 3),
        "spiral_arm_strength": round(target_row["spiral_arm_strength"], 3),
        "edge_on_score": round(target_row["edge_on_score"], 3),
        "vote_fraction": round(target_row["vote_fraction"], 3),
    })

print("Interpretation:")
print("  SP06 is a tilted spiral: its arm signature weakens while edge-on evidence rises.")
print("  EL05 is centrally concentrated, but its flattened shape pushes it toward the edge-on centroid.")
print("  These are exactly the objects that should be handed back to manual review instead of trusted blindly.")


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Optional matplotlib plots skipped: {exc}")
else:
    misclassified_ids = {row["galaxy_id"] for row in misclassified_rows}
    class_colors = {
        "spiral": "tab:blue",
        "elliptical": "tab:orange",
        "edge_on": "tab:green",
    }
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.8))

    for row in analysis_rows:
        marker = "x" if row["galaxy_id"] in misclassified_ids else "o"
        axes[0].scatter(
            row["concentration"],
            row["spiral_arm_strength"],
            color=class_colors[row["morphology_label"]],
            marker=marker,
            s=70,
        )
        axes[1].scatter(
            row["axis_ratio"],
            row["edge_on_score"],
            color=class_colors[row["morphology_label"]],
            marker=marker,
            s=70,
        )
        if row["galaxy_id"] in test_ids:
            axes[0].text(row["concentration"] + 0.02, row["spiral_arm_strength"] + 0.01, row["galaxy_id"], fontsize=8)
            axes[1].text(row["axis_ratio"] + 0.01, row["edge_on_score"] + 0.01, row["galaxy_id"], fontsize=8)

    axes[0].set_xlabel("concentration [unitless]")
    axes[0].set_ylabel("spiral arm strength [unitless]")
    axes[0].set_title("Traditional morphology baseline")

    axes[1].set_xlabel("axis ratio b/a [unitless]")
    axes[1].set_ylabel("edge-on score [unitless]")
    axes[1].set_title("Projection-sensitive feature space")

    plt.tight_layout()
    
    if os.environ.get('AIFA_SKIP_PLOTS') == '1':
        plt.close(figure if 'figure' in locals() else fig)
    else:
        plt.show()


In [ ]:
try:
    from sklearn.neighbors import KNeighborsClassifier
except ModuleNotFoundError as exc:
    print(f"Optional scikit-learn comparison skipped: {exc}")
else:
    classifier = KNeighborsClassifier(n_neighbors=3)
    classifier.fit(
        [row["morphology_vector"] for row in train_rows],
        [row["morphology_label"] for row in train_rows],
    )
    sklearn_predictions = classifier.predict([row["morphology_vector"] for row in test_rows])
    print("sklearn 3-NN predictions:", sklearn_predictions.tolist())


## 导出 Ch29 最小 case package

下面这个单元会把本章结果导出到 `results/ch29_morphology_case/`。重点是保留 baseline、review queue 和 Trust Statement，让后续 CNN 实验站在可复查的前置证据上。

In [ ]:
RESULTS_DIR = Path("results/ch29_morphology_case")
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

review_queue = []
for row in prediction_rows:
    reasons = []
    if row["true_label"] != row["predicted_label"]:
        reasons.append("misclassified")
    if row["distance_margin"] < 0.75:
        reasons.append("low_margin")
    if row["vote_fraction"] < 0.80:
        reasons.append("moderate_vote_fraction")
    if row["axis_ratio"] < 0.50 or row["edge_on_score"] > 0.55:
        reasons.append("projection_ambiguity")
    if reasons:
        review_queue.append({
            "galaxy_id": row["galaxy_id"],
            "true_label": row["true_label"],
            "predicted_label": row["predicted_label"],
            "runner_up_label": row["runner_up_label"],
            "distance_margin": row["distance_margin"],
            "vote_fraction": row["vote_fraction"],
            "axis_ratio": row["axis_ratio"],
            "spiral_arm_strength": row["spiral_arm_strength"],
            "edge_on_score": row["edge_on_score"],
            "review_reasons": ";".join(reasons),
        })

with (RESULTS_DIR / "review_queue.csv").open("w", newline="", encoding="utf-8") as handle:
    fieldnames = [
        "galaxy_id", "true_label", "predicted_label", "runner_up_label", "distance_margin",
        "vote_fraction", "axis_ratio", "spiral_arm_strength", "edge_on_score", "review_reasons",
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(review_queue)

with (RESULTS_DIR / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as handle:
    labels = sorted({row["morphology_label"] for row in analysis_rows})
    writer = csv.DictWriter(handle, fieldnames=["true_label"] + labels)
    writer.writeheader()
    for true_label in labels:
        row = {"true_label": true_label}
        for predicted_label in labels:
            row[predicted_label] = confusion.get(true_label, {}).get(predicted_label, 0)
        writer.writerow(row)

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Figure export skipped: {exc}")
else:
    misclassified_ids = {row["galaxy_id"] for row in misclassified_rows}
    review_ids = {row["galaxy_id"] for row in review_queue}
    class_colors = {
        "spiral": "tab:blue",
        "elliptical": "tab:orange",
        "edge_on": "tab:green",
    }
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    for row in analysis_rows:
        marker = "x" if row["galaxy_id"] in misclassified_ids else "o"
        size = 100 if row["galaxy_id"] in review_ids else 65
        axes[0].scatter(row["concentration"], row["spiral_arm_strength"], color=class_colors[row["morphology_label"]], marker=marker, s=size)
        axes[1].scatter(row["axis_ratio"], row["edge_on_score"], color=class_colors[row["morphology_label"]], marker=marker, s=size)
        if row["galaxy_id"] in review_ids:
            axes[0].text(row["concentration"] + 0.02, row["spiral_arm_strength"] + 0.01, row["galaxy_id"], fontsize=8)
            axes[1].text(row["axis_ratio"] + 0.01, row["edge_on_score"] + 0.01, row["galaxy_id"], fontsize=8)
    axes[0].set_xlabel("concentration")
    axes[0].set_ylabel("spiral arm strength")
    axes[0].set_title("Feature-space diagnostic")
    axes[1].set_xlabel("axis ratio b/a")
    axes[1].set_ylabel("edge-on score")
    axes[1].set_title("Projection and review queue")
    figure.tight_layout()
    figure.savefig(FIGURES_DIR / "feature_space_diagnostic.png", dpi=160)
    plt.close(figure)

(RESULTS_DIR / "dataset_contract_morphology.md").write_text(f"""# Dataset Contract: Ch29 Morphology

## Task Definition
- Scientific question: Can traditional morphology features support a baseline galaxy morphology classifier?
- ML task: classification
- Prediction target: `morphology_label`
- Intended use: teaching-scale baseline before CNN or transfer learning

## Sample Definition
- Sample unit: one galaxy row derived from an image cutout
- Sample ID: `galaxy_id`
- Data file: `{DATA_PATH.relative_to(Path.cwd()) if DATA_PATH.is_relative_to(Path.cwd()) else DATA_PATH.name}`

## Input Features
- Feature set: {', '.join(feature_names)}
- Quality fields: `quality_flag`, `vote_fraction`, `seeing_arcsec`, `snr_per_pixel`
- Derived from: teaching image morphology measurements

## Target / Label
- Target: `morphology_label`
- Label source: teaching Galaxy Zoo-style morphology label
- Label quality: `vote_fraction`
- Ambiguous cases: low vote fraction, projection ambiguity, low margin, misclassification

## Selection / Split
- Quality cut: `quality_flag == clean`
- Vote cut: `vote_fraction >= 0.70`
- Train size: {len(train_rows)}
- Test size: {len(test_rows)}
- Test IDs: {', '.join(row['galaxy_id'] for row in test_rows)}

## Known Limits
- Uses pre-extracted features rather than raw image cutouts
- Small teaching sample
- Projection effects and label noise remain central risks
""", encoding="utf-8")

(RESULTS_DIR / "model_experiment_record_morphology.md").write_text(f"""# Model Experiment Record: Ch29 Morphology

## Task
- Scientific question: Can traditional morphology features separate spiral, elliptical, and edge-on teaching samples?
- ML task: classification
- Prediction target: `morphology_label`

## Dataset Contract
- Dataset Contract link: `dataset_contract_morphology.md`

## Split / Role Assignment
- Split: fixed train/test split
- Test IDs: {', '.join(row['galaxy_id'] for row in test_rows)}

## Baseline
- Baseline type: nearest-centroid classifier on standardized traditional morphology features
- Baseline reason: interpretable pre-CNN baseline
- Test accuracy: {accuracy:.3f}

## Model
- Model family: nearest centroid
- Algorithm Card sidebar links: Ch19 Classification; Ch21 Feature Ledger; Ch20 Evaluation; Ch25 Trust Statement

## Evaluation
- Metrics: accuracy, confusion matrix, recall by class
- Evaluation artifacts: `confusion_matrix.csv`, `figures/feature_space_diagnostic.png`

## Error Analysis
- Error/review artifact: `review_queue.csv`
- Review fields: low margin, misclassification, vote fraction, projection ambiguity

## Limit
- Supported claim: traditional morphology features provide an interpretable baseline on this teaching sample.
- Unsupported claim: real survey morphology classification is solved, or CNN is unnecessary in larger image datasets.
""", encoding="utf-8")

(RESULTS_DIR / "trust_statement_morphology.md").write_text("""# Trust Statement: Ch29 Morphology

## Model Output
- Result: nearest-centroid morphology baseline on traditional features
- Main diagnostic: confusion matrix, feature-space diagnostic, review queue

## Distribution Status
- In-distribution for clean, high-vote teaching samples with similar feature ranges
- Unclear for low-quality, low-vote, low-margin, or projection-ambiguous galaxies

## Uncertainty
- Main source: label noise, projection effects, small sample size, image-quality variation
- Estimated by: vote fraction, margin, misclassification review, and feature-space position

## Interpretability
- Main feature dependence: concentration, asymmetry, axis ratio, spiral-arm strength, edge-on score
- Not causal proof: feature dependence describes this classifier's behavior, not galaxy-formation causality

## Failure Boundary
- Known failure region: low vote fraction, low margin, edge-on projection, low-quality images, neighboring-source contamination
- Human review needed: all rows in `review_queue.csv`

## Claim Boundary
- Supported claim: baseline-before-CNN workflow identifies interpretable morphology evidence and review candidates.
- Unsupported claim: the current teaching baseline can replace human morphology labels or solve full survey morphology classification.
""", encoding="utf-8")

print("Wrote Ch29 case package to", RESULTS_DIR)
print("Review queue:", review_queue)
print("Expected files:", sorted(path.name for path in RESULTS_DIR.iterdir()))